In [63]:
import numpy as np
import torch
import torch.nn as nn
from torch.autograd import Variable
import torchvision
from torch.nn import init

from torchvision.models.resnet import BasicBlock, ResNet
from torchvision.transforms import ToTensor

In [64]:
import io
from torchvision import models, transforms
import torch.utils.data as data_utils
from PIL import Image
import os

import cv2
import matplotlib.pyplot as plt
import torch.nn.functional as F
def default_loader(path):
    return Image.open(path)   

In [65]:
from torchvision.models.resnet import BasicBlock, ResNet
from torch.nn import init

def conv(in_planes, out_planes, kernel_size=3, stride=1, dilation=1, bias=False, transposed=False):
    if transposed:
        layer = nn.ConvTranspose2d(in_planes, out_planes, kernel_size=kernel_size, stride=stride, padding=1, output_padding=1,
                                   dilation=dilation, bias=bias)
    else:
        padding = (kernel_size + 2 * (dilation - 1)) // 2
        layer = nn.Conv2d(in_planes, out_planes, kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation, bias=bias)
    if bias:
        init.constant(layer.bias, 0)
    return layer

# Returns 2D batch normalisation layer
def bn(planes):
    layer = nn.BatchNorm2d(planes)
    # Use mean 0, standard deviation 1 init
    init.constant(layer.weight, 1)
    init.constant(layer.bias, 0)
    return layer


class FeatureResNet(ResNet):
    def __init__(self):
        super().__init__(BasicBlock, [3, 14, 16, 3], 1000)
        self.conv_f = conv(2,64, kernel_size=3,stride = 1)
        self.ReLu_1 = nn.ReLU(inplace=True)
        self.conv_pre = conv(512, 1024, stride=2, transposed=False)
        self.bn_pre = bn(1024)

    def forward(self, x):
        x1 = self.conv_f(x)
        x = self.bn1(x1)
        x = self.relu(x)
        x2 = self.maxpool(x)
        x = self.layer1(x2)
        x3 = self.layer2(x)
        x4 = self.layer3(x3)
        x5 = self.layer4(x4)
        x6 = self.ReLu_1(self.bn_pre(self.conv_pre(x5)))
        return x1, x2, x3, x4, x5,x6


class SegResNet(nn.Module):
    def __init__(self, num_classes, pretrained_net):
        super().__init__()
        self.pretrained_net = pretrained_net
        self.relu = nn.ReLU(inplace=True)
        self.conv3_2 = conv(1024, 512, stride=1, transposed=False)
        self.bn3_2 = bn(512)
        self.conv4 = conv(512,512, stride=2, transposed=True)
        self.bn4 = bn(512)
        self.conv5 = conv(512, 256, stride=2, transposed=True)
        self.bn5 = bn(256)
        self.conv6 = conv(256, 128, stride=2, transposed=True)
        self.bn6 = bn(128)
        self.conv7 = conv(128, 64, stride=2, transposed=True)
        self.bn7 = bn(64)
        self.conv8 = conv(64, 64, stride=2, transposed=True)
        self.bn8 = bn(64)
        self.conv9 = conv(64, 32, stride=2, transposed=True)
        self.bn9 = bn(32)
        self.convadd = conv(32, 16, stride=1, transposed=False)
        self.bnadd = bn(16)
        self.conv10 = conv(16, num_classes,stride=2, kernel_size=5)
        init.constant(self.conv10.weight, 0)  # Zero init

    def forward(self, x):
        
        x1, x2, x3, x4, x5, x6 = self.pretrained_net(x)
        
        x = self.relu(self.bn3_2(self.conv3_2(x6)))
        
        x = self.relu(self.bn4(self.conv4(x)))
        x = self.relu(self.bn5(self.conv5(x)))
        #print(x.size())
        x = self.relu(self.bn6(self.conv6(x+x4 )))
        #print(x.size())
        x = self.relu(self.bn7(self.conv7(x+x3 )))
        #print(x.size())
        x = self.relu(self.bn8(self.conv8(x+x2 )))
        #print(x.size())
        x = self.relu(self.bn9(self.conv9(x+x1 )))
        #print(x.size())
        x = self.relu(self.bnadd(self.convadd(x)))
        x = self.conv10(x)
        return x


In [66]:
fnet = FeatureResNet()
fcn = SegResNet(2,fnet)
fcn = fcn.cpu()


C:\Users\HomePC\AppData\Local\Temp\ipykernel_8300\1245188838.py:19: FutureWarning: `nn.init.constant` is now deprecated in favor of `nn.init.constant_`.
  init.constant(layer.weight, 1)
C:\Users\HomePC\AppData\Local\Temp\ipykernel_8300\1245188838.py:20: FutureWarning: `nn.init.constant` is now deprecated in favor of `nn.init.constant_`.
  init.constant(layer.bias, 0)
C:\Users\HomePC\AppData\Local\Temp\ipykernel_8300\1245188838.py:67: FutureWarning: `nn.init.constant` is now deprecated in favor of `nn.init.constant_`.
  init.constant(self.conv10.weight, 0)  # Zero init


In [67]:
dataset_path = '.'

# Validation
test_set = []

for i in range(4000):
    test_set.append((
        dataset_path + '/imgs3/train_image_' + str(i+1) + '_1.png',
        dataset_path + '/imgs3/train_image_' + str(i+1) + '_2.png',
        dataset_path + '/gt3/train_image_' + str(i+1) + '.mat'
    ))

# Training
train_set = []

for i in range(36000):
    train_set.append((
        dataset_path + '/imgs3/train_image_' + str(i+1) + '_1.png',
        dataset_path + '/imgs3/train_image_' + str(i+1) + '_2.png',
        dataset_path + '/gt3/train_image_' + str(i+1) + '.mat'
    ))


In [68]:
import torch
import numpy as np
import scipy.io as sio
from torchvision.transforms import ToTensor
from PIL import Image

# 1. Define the default_loader if it's not already defined
def default_loader(path):
    return Image.open(path).convert('RGB')

# 2. Corrected Dataset Class
class MyDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, transform=None, target_transform=None, loader=default_loader):
        self.imgs = dataset
        self.transform = transform
        self.target_transform = target_transform
        self.loader = loader

    def __getitem__(self, index):
        label_x, label_y, label_z = self.imgs[index]
        
        # Load and resize images
        img1 = self.loader(label_x)
        img_1 = ToTensor()(img1.resize((128, 128)))
        
        img2 = self.loader(label_y)
        img_2 = ToTensor()(img_2.resize((128, 128)) if 'img_2' in locals() else img2.resize((128, 128)))
        
        # Stack images along the channel dimension
        imgs = torch.cat((img_1, img_2), 0)
        
        try:
            # Load .mat file using scipy.io
            mat_data = sio.loadmat(label_z)
            if 'Disp_field_1' in mat_data:
                gt = mat_data['Disp_field_1'].astype(float)
            else:
                gt = mat_data['Disp_field_2'].astype(float)
                
        except Exception as e:
            print(f"Error loading mat file at {label_z}: {e}")
            # Return a zero tensor or handle as needed
            gt = np.zeros((128, 128, 2)) 

        # Downsample and reorder axes for PyTorch (HWC -> CHW)
        gt = gt[::2, ::2, :]
        gt = np.moveaxis(gt, -1, 0)
        
        return imgs, torch.from_numpy(gt).float()

    def __len__(self):
        return len(self.imgs)

In [69]:
EPOCH = 100              # train the training data n times, to save time, we just train 1 epoch
BATCH_SIZE = 12
print('BATCH_SIZE = ',BATCH_SIZE)
LR = 0.001              # learning rate
#root = './gdrive_northwestern/My Drive/dl_encoder/data/orig/orig'
NUM_WORKERS = 0

optimizer = torch.optim.Adam(fcn.parameters(), lr=LR)   # optimize all cnn parameters
#optimizer = torch.optim.SGD(cnn.parameters(), lr=LR, momentum=0.9)   # optimize all cnn parameters
loss_func = nn.MSELoss()


train_data=MyDataset(dataset=train_set)
train_loader = data_utils.DataLoader(dataset=train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

test_data=MyDataset(dataset=test_set)
test_loader = data_utils.DataLoader(dataset=test_data, batch_size=1)

BATCH_SIZE =  12


In [70]:
from datetime import datetime
dataString = datetime.strftime(datetime.now(), '%Y_%m_%d_%H:%M:%S')

In [71]:
import os
from pathlib import Path

# 1. Define a valid name for your root results folder
# Using '.' refers to the current working directory
root_result = 'results' 

# 2. Use Path to create the directories safely
# parents=True: creates any missing parent folders
# exist_ok=True: doesn't crash if the folder already exists
Path(root_result).mkdir(parents=True, exist_ok=True)

model_result = os.path.join(root_result, 'model')
log_result = os.path.join(root_result, 'log')

# 3. Create the subdirectories
os.makedirs(model_result, exist_ok=True)
os.makedirs(log_result, exist_ok=True)

print(f"Directories created at: {os.path.abspath(root_result)}")

Directories created at: c:\Users\HomePC\Documents\GitHub\Deep-Dic-deep-learning-based-digital-image-correlation\results


In [ ]:
import torch
import datetime
from torch.autograd import Variable

# Corrected dataString for Windows compatibility
dataString = datetime.datetime.now().strftime("%Y_%m_%d_%H_%M_%S")

# Initialize log files
fileOut = open(log_result + 'log_' + dataString + '.txt', 'a')
fileOut.write(f'{dataString} Epoch:   Step:    Loss:        Val_Accu :\n')
fileOut.close()

fileOut2 = open(log_result + 'validation_' + dataString + '.txt', 'a')
fileOut2.write('kernel_size of conv_f is 2\n')
fileOut2.write(f'{dataString} Epoch:    loss:\n')
fileOut2.close()

for epoch in range(EPOCH):
    fcn.train()
    dataset_path = '.'
    imgs_dir = os.path.join(dataset_path, 'imgs3')
    gt_dir = os.path.join(dataset_path, 'gt3')

    all_img1 = sorted([f for f in os.listdir(imgs_dir) if f.endswith('_1.png')])

    test_set = []
    train_set = []

    for img1_name in all_img1:
        idx = img1_name[len('train_image_'):-len('_1.png')]
        img1_path = os.path.join(imgs_dir, img1_name)
        img2_path = os.path.join(imgs_dir, f'train_image_{idx}_2.png')
        mat_path = os.path.join(gt_dir, f'train_image_{idx}.mat')

        if not (os.path.exists(img1_path) and os.path.exists(img2_path) and os.path.exists(mat_path)):
            print('Skipping missing file triplet:', img1_path, img2_path, mat_path)
            continue

        if len(test_set) < 4000:
            test_set.append((img1_path, img2_path, mat_path))
        else:
            train_set.append((img1_path, img2_path, mat_path))

    print('train set length:', len(train_set))
    print('test set length:', len(test_set))
        # CPU FIX: Removed .cuda()
        # Corrected dataString for Windows compatibility
        # Corrected dataString for Windows compatibility
        # Corrected dataString for Windows compatibility
        # Corrected dataString for Windows compatibility
        dataString = datetime.datetime.now().strftime("%Y_%m_%d_%H_%M_%S")

        # Initialize log files
        fileOut = open(log_result + 'log_' + dataString + '.txt', 'a')
        fileOut.write(f'{dataString} Epoch:   Step:    Loss:        Val_Accu :\n')
        fileOut.close()

        fileOut2 = open(log_result + 'validation_' + dataString + '.txt', 'a')
        fileOut2.write('kernel_size of conv_f is 2\n')
        fileOut2.write(f'{dataString} Epoch:    loss:\n')
        fileOut2.close()

        for epoch in range(EPOCH):
            fcn.train()
            dataset_path = '.'
            imgs_dir = os.path.join(dataset_path, 'imgs3')
            gt_dir = os.path.join(dataset_path, 'gt3')

            all_img1 = sorted([f for f in os.listdir(imgs_dir) if f.endswith('_1.png')])

            test_set = []
            train_set = []

            for img1_name in all_img1:
                idx = img1_name[len('train_image_'):-len('_1.png')]
                img1_path = os.path.join(imgs_dir, img1_name)
                img2_path = os.path.join(imgs_dir, f'train_image_{idx}_2.png')
                mat_path = os.path.join(gt_dir, f'train_image_{idx}.mat')

                if not (os.path.exists(img1_path) and os.path.exists(img2_path) and os.path.exists(mat_path)):
                    print('Skipping missing file triplet:', img1_path, img2_path, mat_path)
                    continue

                if len(test_set) < 4000:
                    test_set.append((img1_path, img2_path, mat_path))
                else:
                    train_set.append((img1_path, img2_path, mat_path))

            print('train set length:', len(train_set))
            print('test set length:', len(test_set))
            dataString = datetime.datetime.now().strftime("%Y_%m_%d_%H_%M_%S")

            # Recreate train_loader for this epoch (assuming shuffling or updates)
            train_data = MyDataset(dataset=train_set)
            train_loader = data_utils.DataLoader(dataset=train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

            for step, (img, gt) in enumerate(train_loader):
                # CPU FIX: Removed .cuda()
                img = img.float()
                gt = gt.float()
                
                output = fcn(img)
                loss = loss_func(output, gt)
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                current_loss = loss.item()
                print(f"Epoch: {epoch} | Step: {step} | Loss: {current_loss}")
                
                # Log training progress
                with open(log_result + 'log_' + dataString + '.txt', 'a') as f:
                    f.write(f"{epoch}   {step}   {current_loss}\n")

            # Save Checkpoints
            if epoch % 10 == 9:
                PATH = model_result + f'param_all_2_{epoch}_{step}.pth'
                torch.save(fcn.state_dict(), PATH)
                print('Finished saving checkpoint')

            # Validation Phase
            LOSS_VALIDATION = 0
            fcn.eval()
            with torch.no_grad():
                for step_val, (img, gt) in enumerate(test_loader):
                    # CPU FIX: Removed .cuda()
                    img = img.float()
                    gt = gt.float() 
                    
                    output = fcn(img)
                    # Ensure output and gt shapes match for loss function
                    LOSS_VALIDATION += loss_func(output, gt).item()
                
                avg_val_loss = LOSS_VALIDATION / (step_val + 1)
                
                # Log validation results
                with open(log_result + 'validation_' + dataString + '.txt', 'a') as f:
                    f.write(f"{epoch}   {step_val}   {avg_val_loss}\n")
                    
                print(f'Validation error epoch {epoch}: {avg_val_loss}')

        # Initialize log files
        fileOut = open(log_result + 'log_' + dataString + '.txt', 'a')
        fileOut.write(f'{dataString} Epoch:   Step:    Loss:        Val_Accu :\n')
        fileOut.close()

        fileOut2 = open(log_result + 'validation_' + dataString + '.txt', 'a')
        fileOut2.write('kernel_size of conv_f is 2\n')
        fileOut2.write(f'{dataString} Epoch:    loss:\n')
        fileOut2.close()

        for epoch in range(EPOCH):
            fcn.train()
            dataset_path = '.'
            imgs_dir = os.path.join(dataset_path, 'imgs3')
            gt_dir = os.path.join(dataset_path, 'gt3')

            all_img1 = sorted([f for f in os.listdir(imgs_dir) if f.endswith('_1.png')])

            test_set = []
            train_set = []

            for img1_name in all_img1:
                idx = img1_name[len('train_image_'):-len('_1.png')]
                img1_path = os.path.join(imgs_dir, img1_name)
                img2_path = os.path.join(imgs_dir, f'train_image_{idx}_2.png')
                mat_path = os.path.join(gt_dir, f'train_image_{idx}.mat')

                if not (os.path.exists(img1_path) and os.path.exists(img2_path) and os.path.exists(mat_path)):
                    print('Skipping missing file triplet:', img1_path, img2_path, mat_path)
                    continue

                if len(test_set) < 4000:
                    test_set.append((img1_path, img2_path, mat_path))
                else:
                    train_set.append((img1_path, img2_path, mat_path))

            print('train set length:', len(train_set))
            print('test set length:', len(test_set))

            # Recreate train_loader for this epoch (assuming shuffling or updates)
            train_data = MyDataset(dataset=train_set)
            train_loader = data_utils.DataLoader(dataset=train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

            for step, (img, gt) in enumerate(train_loader):
                # CPU FIX: Removed .cuda()
                img = img.float()
                gt = gt.float()
                
                output = fcn(img)
                loss = loss_func(output, gt)
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                current_loss = loss.item()
                print(f"Epoch: {epoch} | Step: {step} | Loss: {current_loss}")
                
                # Log training progress
                with open(log_result + 'log_' + dataString + '.txt', 'a') as f:
                    f.write(f"{epoch}   {step}   {current_loss}\n")

            # Save Checkpoints
            if epoch % 10 == 9:
                PATH = model_result + f'param_all_2_{epoch}_{step}.pth'
                torch.save(fcn.state_dict(), PATH)
                print('Finished saving checkpoint')

            # Validation Phase
            LOSS_VALIDATION = 0
            fcn.eval()
            with torch.no_grad():
                for step_val, (img, gt) in enumerate(test_loader):
                    # CPU FIX: Removed .cuda()
                    img = img.float()
                    gt = gt.float() 
                    
                    output = fcn(img)
                    # Ensure output and gt shapes match for loss function
                    LOSS_VALIDATION += loss_func(output, gt).item()
                
                avg_val_loss = LOSS_VALIDATION / (step_val + 1)
                
                # Log validation results
                with open(log_result + 'validation_' + dataString + '.txt', 'a') as f:
                    f.write(f"{epoch}   {step_val}   {avg_val_loss}\n")
                    
                print(f'Validation error epoch {epoch}: {avg_val_loss}')

        # Initialize log files
        fileOut = open(log_result + 'log_' + dataString + '.txt', 'a')
        fileOut.write(f'{dataString} Epoch:   Step:    Loss:        Val_Accu :\n')
        fileOut.close()

        fileOut2 = open(log_result + 'validation_' + dataString + '.txt', 'a')
        fileOut2.write('kernel_size of conv_f is 2\n')
        fileOut2.write(f'{dataString} Epoch:    loss:\n')
        fileOut2.close()

        for epoch in range(EPOCH):
            fcn.train()
            dataset_path = '.'
            imgs_dir = os.path.join(dataset_path, 'imgs3')
            gt_dir = os.path.join(dataset_path, 'gt3')

            all_img1 = sorted([f for f in os.listdir(imgs_dir) if f.endswith('_1.png')])

            test_set = []
            train_set = []

            for img1_name in all_img1:
                idx = img1_name[len('train_image_'):-len('_1.png')]
                img1_path = os.path.join(imgs_dir, img1_name)
                img2_path = os.path.join(imgs_dir, f'train_image_{idx}_2.png')
                mat_path = os.path.join(gt_dir, f'train_image_{idx}.mat')

                if not (os.path.exists(img1_path) and os.path.exists(img2_path) and os.path.exists(mat_path)):
                    print('Skipping missing file triplet:', img1_path, img2_path, mat_path)
                    continue

                if len(test_set) < 4000:
                    test_set.append((img1_path, img2_path, mat_path))
                else:
                    train_set.append((img1_path, img2_path, mat_path))

            print('train set length:', len(train_set))
            print('test set length:', len(test_set))

            # Recreate train_loader for this epoch (assuming shuffling or updates)
            train_data = MyDataset(dataset=train_set)
            train_loader = data_utils.DataLoader(dataset=train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

            for step, (img, gt) in enumerate(train_loader):
                # CPU FIX: Removed .cuda()
                img = img.float()
                gt = gt.float()
                
                output = fcn(img)
                loss = loss_func(output, gt)
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                current_loss = loss.item()
                print(f"Epoch: {epoch} | Step: {step} | Loss: {current_loss}")
                
                # Log training progress
                with open(log_result + 'log_' + dataString + '.txt', 'a') as f:
                    f.write(f"{epoch}   {step}   {current_loss}\n")

            # Save Checkpoints
            if epoch % 10 == 9:
                PATH = model_result + f'param_all_2_{epoch}_{step}.pth'
                torch.save(fcn.state_dict(), PATH)
                print('Finished saving checkpoint')

            # Validation Phase
            LOSS_VALIDATION = 0
            fcn.eval()
            with torch.no_grad():
                for step_val, (img, gt) in enumerate(test_loader):
                    # CPU FIX: Removed .cuda()
                    img = img.float()
                    gt = gt.float() 
                    
                    output = fcn(img)
                    # Ensure output and gt shapes match for loss function
                    LOSS_VALIDATION += loss_func(output, gt).item()
                
                avg_val_loss = LOSS_VALIDATION / (step_val + 1)
                
                # Log validation results
                with open(log_result + 'validation_' + dataString + '.txt', 'a') as f:
                    f.write(f"{epoch}   {step_val}   {avg_val_loss}\n")
                    
                print(f'Validation error epoch {epoch}: {avg_val_loss}')

        # Initialize log files
        fileOut = open(log_result + 'log_' + dataString + '.txt', 'a')
        fileOut.write(f'{dataString} Epoch:   Step:    Loss:        Val_Accu :\n')
        fileOut.close()

        fileOut2 = open(log_result + 'validation_' + dataString + '.txt', 'a')
        fileOut2.write('kernel_size of conv_f is 2\n')
        fileOut2.write(f'{dataString} Epoch:    loss:\n')
        fileOut2.close()

        for epoch in range(EPOCH):
            fcn.train()
            dataset_path = '.'
            imgs_dir = os.path.join(dataset_path, 'imgs3')
            gt_dir = os.path.join(dataset_path, 'gt3')

            all_img1 = sorted([f for f in os.listdir(imgs_dir) if f.endswith('_1.png')])

            test_set = []
            train_set = []

            for img1_name in all_img1:
                idx = img1_name[len('train_image_'):-len('_1.png')]
                img1_path = os.path.join(imgs_dir, img1_name)
                img2_path = os.path.join(imgs_dir, f'train_image_{idx}_2.png')
                mat_path = os.path.join(gt_dir, f'train_image_{idx}.mat')

                if not (os.path.exists(img1_path) and os.path.exists(img2_path) and os.path.exists(mat_path)):
                    print('Skipping missing file triplet:', img1_path, img2_path, mat_path)
                    continue

                if len(test_set) < 4000:
                    test_set.append((img1_path, img2_path, mat_path))
                else:
                    train_set.append((img1_path, img2_path, mat_path))

            print('train set length:', len(train_set))
            print('test set length:', len(test_set))

            # Recreate train_loader for this epoch (assuming shuffling or updates)
            train_data = MyDataset(dataset=train_set)
            train_loader = data_utils.DataLoader(dataset=train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

            for step, (img, gt) in enumerate(train_loader):
                # CPU FIX: Removed .cuda()
                img = img.float()
                gt = gt.float()
                
                output = fcn(img)
                loss = loss_func(output, gt)
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                current_loss = loss.item()
                print(f"Epoch: {epoch} | Step: {step} | Loss: {current_loss}")
                
                # Log training progress
                with open(log_result + 'log_' + dataString + '.txt', 'a') as f:
                    f.write(f"{epoch}   {step}   {current_loss}\n")

            # Save Checkpoints
            if epoch % 10 == 9:
                PATH = model_result + f'param_all_2_{epoch}_{step}.pth'
                torch.save(fcn.state_dict(), PATH)
                print('Finished saving checkpoint')

            # Validation Phase
            LOSS_VALIDATION = 0
            fcn.eval()
            with torch.no_grad():
                for step_val, (img, gt) in enumerate(test_loader):
                    # CPU FIX: Removed .cuda()
                    img = img.float()
                    gt = gt.float() 
                    
                    output = fcn(img)
                    # Ensure output and gt shapes match for loss function
                    LOSS_VALIDATION += loss_func(output, gt).item()
                
                avg_val_loss = LOSS_VALIDATION / (step_val + 1)
                
                # Log validation results
                with open(log_result + 'validation_' + dataString + '.txt', 'a') as f:
                    f.write(f"{epoch}   {step_val}   {avg_val_loss}\n")
                    
                print(f'Validation error epoch {epoch}: {avg_val_loss}')
        gt = gt.float()
        
        output = fcn(img)
        loss = loss_func(output, gt)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        current_loss = loss.item()
        print(f"Epoch: {epoch} | Step: {step} | Loss: {current_loss}")
        
        # Log training progress
        with open(log_result + 'log_' + dataString + '.txt', 'a') as f:
            f.write(f"{epoch}   {step}   {current_loss}\n")

    # Save Checkpoints
    if epoch % 10 == 9:
        PATH = model_result + f'param_all_2_{epoch}_{step}.pth'
        torch.save(fcn.state_dict(), PATH)
        print('Finished saving checkpoint')

    # Validation Phase
    LOSS_VALIDATION = 0
    fcn.eval()
    with torch.no_grad():
        for step_val, (img, gt) in enumerate(test_loader):
            # CPU FIX: Removed .cuda()
            img = img.float()
            gt = gt.float() 
            
            output = fcn(img)
            # Ensure output and gt shapes match for loss function
            LOSS_VALIDATION += loss_func(output, gt).item()
        
        avg_val_loss = LOSS_VALIDATION / (step_val + 1)
        
        # Log validation results
        with open(log_result + 'validation_' + dataString + '.txt', 'a') as f:
            f.write(f"{epoch}   {step_val}   {avg_val_loss}\n")
            
        print(f'Validation error epoch {epoch}: {avg_val_loss}')

IndentationError: unexpected indent (1115643039.py, line 50)

In [25]:
import datetime

# INSTEAD OF: dataString = str(datetime.datetime.now()) 
# USE THIS:
dataString = datetime.datetime.now().strftime("%Y_%m_%d_%H_%M_%S")